# Cierre de la semana de imágenes: ¿qué más puede "ver" la IA?


En la clase anterior usamos YOLO para **detectar objetos**. Hoy hacemos dos cosas:

1. **Recorrido por tareas (25 min):** aplicamos cinco tareas distintas a la misma imagen y comparamos qué "ve" cada modelo.
2. **CLIP, tú pones las categorías (20 min):** clasificamos imágenes con etiquetas que escribimos nosotros y vemos cómo cambian los resultados según las palabras que elegimos.
3. **Cierre (10 min):** discusión en grupo.

Una idea para tener presente durante toda la clase: **cada modelo responde a una pregunta distinta sobre la imagen**. Ninguno "entiende" la imagen como lo haría una persona.

## 0. Preparación

Instalamos las librerías que no vienen en Colab. Esta celda puede tardar uno o dos minutos; mientras tanto, revisemos el plan de la clase.

In [ ]:
!pip install -q ultralytics easyocr

In [ ]:
import numpy as np
import requests
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
from google.colab import files
from ultralytics import YOLO
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import pipeline

Creamos una función para **mostrar dos imágenes lado a lado**: la original y el resultado del modelo. La vamos a usar varias veces, por eso conviene escribirla una sola vez.

In [ ]:
def mostrar_lado_a_lado(imagen_original, imagen_resultado, titulo_resultado):
    figura, ejes = plt.subplots(1, 2, figsize=(12, 6))

    ejes[0].imshow(imagen_original)
    ejes[0].set_title("Imagen original")
    ejes[0].axis("off")

    ejes[1].imshow(imagen_resultado)
    ejes[1].set_title(titulo_resultado)
    ejes[1].axis("off")

    plt.show()

### Cargar una imagen

Tienen dos opciones. Usen **solo una** de las dos celdas siguientes.

**Opción A:** una imagen de ejemplo desde internet.

In [ ]:
direccion_imagen = "https://ultralytics.com/images/bus.jpg"

respuesta = requests.get(direccion_imagen)
imagen = Image.open(BytesIO(respuesta.content)).convert("RGB")

plt.imshow(imagen)
plt.axis("off")
plt.show()

**Opción B:** subir una foto propia. Busquen una imagen con **personas y algo de texto** (un letrero, un cartel, una pancarta) para que todas las tareas tengan algo que mostrar.

In [ ]:


archivos_subidos = files.upload()
nombre_archivo = list(archivos_subidos.keys())[0]

imagen = Image.open(nombre_archivo).convert("RGB")

# Las fotos de celular suelen ser muy grandes; las reducimos para que todo corra más rápido
imagen.thumbnail((1024, 1024))

plt.imshow(imagen)
plt.axis("off")
plt.show()

---
# Parte 1. Recorrido por tareas

Vamos a aplicar cinco modelos a **la misma imagen**. Después de cada uno, anoten en una línea: **¿qué pregunta responde este modelo?**

| Tarea | Pregunta que responde |
|---|---|
| Detección (clase anterior) | ¿Qué objetos hay y dónde están? |
| Pose | |
| Segmentación | |
| Profundidad | |
| Descripción (captioning) | |
| OCR | |

## 1.1 Estimación de pose

Usamos otra versión de YOLO, entrenada para encontrar **puntos del cuerpo** (cabeza, hombros, codos, rodillas). Fíjense que el código es casi igual al de la clase anterior: solo cambia el nombre del modelo.

In [ ]:


modelo_pose = YOLO("yolo11n-pose.pt")
resultados_pose = modelo_pose(imagen)

# plot() devuelve la imagen en formato BGR; [:, :, ::-1] la convierte a RGB para mostrarla bien
imagen_pose = resultados_pose[0].plot()[:, :, ::-1]

mostrar_lado_a_lado(imagen, imagen_pose, "Pose")

**Para pensar:** ¿para qué se usa la estimación de pose? (deporte, rehabilitación, animación... y también vigilancia y análisis de comportamiento en espacios públicos).

## 1.2 Segmentación

La detección dibuja **cajas**. La segmentación marca **la forma exacta** de cada objeto, píxel por píxel.

In [ ]:
modelo_segmentacion = YOLO("yolo11n-seg.pt")
resultados_segmentacion = modelo_segmentacion(imagen)

imagen_segmentada = resultados_segmentacion[0].plot()[:, :, ::-1]

mostrar_lado_a_lado(imagen, imagen_segmentada, "Segmentación")

**Para pensar:** las categorías son las mismas 80 clases de COCO que vimos con YOLO. ¿Qué cosas de la imagen quedaron sin segmentar porque no existen en esa lista?

## 1.3 Estimación de profundidad

Este modelo intenta adivinar **qué está cerca y qué está lejos** a partir de una sola foto. Los tonos claros son lo que el modelo cree que está más cerca.

Aquí usamos `pipeline` de Hugging Face: una forma rápida de cargar un modelo ya entrenado indicando solo la tarea y el nombre del modelo.

In [ ]:


estimador_profundidad = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")
resultado_profundidad = estimador_profundidad(imagen)

mapa_profundidad = resultado_profundidad["depth"]

mostrar_lado_a_lado(imagen, mapa_profundidad, "Profundidad estimada")

## 1.4 Descripción automática (captioning)

Este modelo genera **una frase** que describe la imagen. Es un primer ejemplo de modelo que conecta **imagen y texto**.

In [ ]:


nombre_modelo_blip = "Salesforce/blip-image-captioning-base"
procesador_blip = BlipProcessor.from_pretrained(nombre_modelo_blip)
modelo_blip = BlipForConditionalGeneration.from_pretrained(nombre_modelo_blip)

entradas = procesador_blip(images=imagen, return_tensors="pt")
salida = modelo_blip.generate(**entradas, max_new_tokens=30)
descripcion = procesador_blip.decode(salida[0], skip_special_tokens=True)

print("Descripción generada:", descripcion)

**Para pensar:**
- ¿En qué idioma respondió? ¿Por qué creen que pasa eso?
- ¿La descripción es neutral? ¿Qué decidió mencionar y qué dejó fuera?
- Imaginen que esta descripción se usa como texto alternativo para personas ciegas. ¿Sería suficiente?

## 1.5 Reconocimiento de texto (OCR)

El OCR "lee" el texto que aparece en una imagen. Es muy útil para digitalizar archivos, prensa o documentos; lo veremos a fondo en la semana de documentos.

Le indicamos que busque texto en español y en inglés.

In [ ]:
import easyocr

In [ ]:


lector_ocr = easyocr.Reader(["es", "en"])
resultados_ocr = lector_ocr.readtext(np.array(imagen))

print("Textos encontrados:", len(resultados_ocr))

Cada resultado trae tres cosas: **la caja** donde está el texto, **el texto** leído y **la confianza** del modelo (de 0 a 1).

Recorremos la lista con un `for`. En cada vuelta, Python toma un resultado y separa sus tres partes en tres variables.

In [ ]:
for caja, texto_leido, confianza in resultados_ocr:
    print(texto_leido, "| confianza:", round(confianza, 2))

**Para pensar:** ¿qué textos leyó bien y cuáles no? ¿Cómo creen que le iría con letra manuscrita, un documento antiguo o una lengua indígena?

### Actividad de la Parte 1 (en grupos, 5 min)

Cada grupo elige **una** de las cinco tareas y responde:

1. Un uso posible en su campo de investigación.
2. Un riesgo o limitación de usarla en ese contexto.

---
# Parte 2. CLIP: tú pones las categorías

Hasta ahora, los modelos solo podían responder con **categorías fijas** (las 80 clases de COCO, por ejemplo).

**CLIP** funciona distinto: aprendió a relacionar imágenes con frases, así que podemos **escribir nosotros las etiquetas** que queramos, sin reentrenar nada. A esto se le llama clasificación *zero-shot*.

Esto parece muy flexible... pero también traslada una decisión importante a quien usa el modelo: **¿con qué palabras nombramos lo que vemos?**

In [ ]:
clasificador_clip = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

Creamos una función que clasifica una imagen con una lista de etiquetas y muestra los resultados en un gráfico de barras.

In [ ]:
def clasificar_y_graficar(imagen, etiquetas, plantilla="This is a photo of {}."):
    resultados = clasificador_clip(imagen, candidate_labels=etiquetas, hypothesis_template=plantilla)

    nombres = []
    puntajes = []
    for resultado in resultados:
        nombres.append(resultado["label"])
        puntajes.append(resultado["score"])

    figura, ejes = plt.subplots(1, 2, figsize=(13, 4))

    ejes[0].imshow(imagen)
    ejes[0].axis("off")

    ejes[1].barh(nombres, puntajes)
    ejes[1].invert_yaxis()
    ejes[1].set_xlim(0, 1)
    ejes[1].set_xlabel("Probabilidad según CLIP")

    plt.tight_layout()
    plt.show()

## 2.1 Primera prueba

Empecemos con etiquetas sencillas sobre la imagen que ya tenemos cargada.

In [ ]:
etiquetas = ["a bus", "a street", "people walking", "a market"]

clasificar_y_graficar(imagen, etiquetas)

**Importante:** las probabilidades **siempre suman 1** entre las etiquetas que le damos. CLIP no puede decir "ninguna de estas": siempre elige alguna.

**Prueben:** pongan cuatro etiquetas que no tengan nada que ver con la imagen (por ejemplo, "a cat", "a volcano", "a book", "a guitar"). ¿Qué pasa?

In [ ]:
# Escriban aquí sus etiquetas
etiquetas = ["a cat", "a volcano", "a book", "a guitar"]

clasificar_y_graficar(imagen, etiquetas)

## 2.2 ¿Habla español?

Por defecto, el modelo arma frases en inglés: *"This is a photo of {etiqueta}."* Si le damos etiquetas en español, quedan frases mezcladas como *"This is a photo of un mercado."*

Comparen tres versiones con las **mismas ideas**:
1. Etiquetas en inglés con la plantilla en inglés.
2. Etiquetas en español con la plantilla en inglés.
3. Etiquetas en español con una plantilla en español.

In [ ]:
etiquetas_ingles = ["a bus", "a street", "people walking", "a market"]
etiquetas_espanol = ["un bus", "una calle", "personas caminando", "un mercado"]

clasificar_y_graficar(imagen, etiquetas_ingles)
clasificar_y_graficar(imagen, etiquetas_espanol)
clasificar_y_graficar(imagen, etiquetas_espanol, plantilla="Esta es una foto de {}.")

**Para pensar:** CLIP se entrenó con cientos de millones de pares imagen-texto tomados de internet, en su mayoría en inglés. ¿Qué implica eso para quienes investigan en español, kichwa u otras lenguas?

## 2.3 Ejercicio principal: las palabras que elegimos

Cada grupo sube **una imagen** relacionada con su tema de investigación (una manifestación, un mercado, un barrio, una fiesta, un espacio de trabajo...).

Luego prueban **dos o tres listas de etiquetas** que describan lo mismo con palabras distintas. Algunos ejemplos:

| Lista 1 | Lista 2 |
|---|---|
| a peaceful protest | a riot |
| a street vendor | informal commerce |
| a working-class neighborhood | a slum |
| a community celebration | a crowd |

Escriban el código ustedes mismos usando la función `clasificar_y_graficar`.

In [ ]:
# 1. Suban su imagen (pueden copiar el código de la Opción B del inicio)


# 2. Escriban sus listas de etiquetas


# 3. Llamen a clasificar_y_graficar con cada lista

**Preguntas para el grupo:**

1. ¿Cambió la etiqueta "ganadora" según las palabras que eligieron?
2. Si una institución usara CLIP para clasificar miles de fotos (por ejemplo, de redes sociales o de cámaras públicas), ¿quién decidiría las etiquetas? ¿Con qué consecuencias?
3. ¿Qué tipo de imágenes y descripciones creen que vio CLIP durante su entrenamiento? ¿Qué mundos quedan fuera?


# Cierre

Piensen n esto:

- **Algo que la IA "ve" bien** en imágenes.
- **Algo que no puede ver**, o que ve desde una perspectiva particular.

**Resumen de la semana:** detección, pose, segmentación, profundidad, descripción, OCR y clasificación zero-shot. Todos estos modelos convierten una imagen en números y responden una pregunta concreta, pero **las preguntas, las categorías y los datos de entrenamiento los decidió alguien**.

**Próxima semana:** documentos. Retomaremos el OCR para trabajar con documentos escaneados.